In [4]:
# Day 10

import sqlite3
import pandas as pd
import json

# 1. Connect to the database (since we are in the ml_model folder, we step back one directory to reach the backend folder)
db_path = '../backend/biometrics.db' 
conn = sqlite3.connect(db_path)

# 2. Extract user data
# ⚠️ VERY IMPORTANT: Change 'your_username' to the actual name you registered with in the frontend
query = "SELECT username, biometric_profile FROM users WHERE username = 'test'"
df_raw = pd.read_sql_query(query, conn)
conn.close()

# 3. Convert complex data (JSON) into a simplified tabular format (DataFrame)
if not df_raw.empty:
    raw_json = df_raw.iloc[0]['biometric_profile']
    profile_data = json.loads(raw_json)
    
    structured_data = []
    
    # Iterate through the five attempts
    for i, attempt in enumerate(profile_data):
        row = {'Attempt_Number': i + 1}
        
        # Extract Dwell Times (DT)
        for dt_obj in attempt['dwellTimes']:
            row[f"DT_{dt_obj['key']}"] = dt_obj['dt']
            
        # Extract Flight Times (FT)
        for ft_obj in attempt['flightTimes']:
            row[f"FT_{ft_obj['transition']}"] = ft_obj['ft']
            
        # Extract total duration
        row['Total_Duration'] = attempt.get('totalDuration', 0)
        
        structured_data.append(row)

    # 4. Create the final dataframe
    df_features = pd.DataFrame(structured_data)
    
    print("=== Data successfully loaded into Pandas DataFrame ===")
    display(df_features) # This function prints the table elegantly in Jupyter
else:
    print("Sorry, user not found or no data registered under this name.")

=== Data successfully loaded into Pandas DataFrame ===


,Attempt_Number,DT_t,DT_e,DT_s,DT_p,DT_a,FT_t -> e,FT_e -> s,FT_s -> t,FT_t -> p,FT_p -> a,FT_a -> s,FT_s -> s,FT_s -> Enter,Total_Duration,DT_o,DT_Backspace,FT_t -> o,FT_o -> Backspace,FT_Backspace -> p
0,1,80.2,95.5,87.9,79.5,95.2,73.0,135.9,111.9,800.2,72.6,152.7,88.1,160.5,2143.8,NaN,NaN,NaN,NaN,NaN
1,2,87.5,87.6,72.7,72.2,95.7,64.3,135.9,88.2,896.1,56.2,136.0,104.3,179.1,2127.9,NaN,NaN,NaN,NaN,NaN
2,3,111.6,87.7,63.6,87.7,70.7,144.4,144.1,200.3,NaN,296.7,152.9,80.0,736.4,2927.5,87.4,71.6,296.4,616.4,176.8
3,4,95.3,119.4,80.2,88.7,126.6,167.6,160.6,240.6,448.1,168.4,104.6,80.4,176.4,2183.6,NaN,NaN,NaN,NaN,NaN
4,5,96.1,135.2,71.9,80.4,144.1,56.5,-15.2,48.0,296.1,7.0,-15.7,31.4,200.2,1239.8,NaN,NaN,NaN,NaN,NaN
5,6,88.4,232.1,73.4,95.5,118.2,-71.5,0.3,463.8,24.0,73.9,88.5,71.6,151.4,1696.0,NaN,NaN,NaN,NaN,NaN
6,7,78.8,152.1,80.0,79.7,152.2,463.5,-39.7,705.2,64.7,24.0,-40.2,88.5,63.7,2103.6,NaN,NaN,NaN,NaN,NaN
7,8,64.0,143.4,63.5,71.5,127.6,88.5,-23.6,152.2,224.3,16.1,-39.7,72.1,112.5,1262.4,NaN,NaN,NaN,NaN,NaN
8,9,63.6,159.7,71.9,86.9,143.2,119.8,-31.4,176.8,392.8,-6.8,-23.7,87.9,56.2,1536.2,NaN,NaN,NaN,NaN,NaN
9,10,103.6,137.5,71.9,88.0,143.5,64.2,-41.3,32.0,368.4,48.2,-31.7,72.0,79.8,1423.5,NaN,NaN,NaN,NaN,NaN


In [5]:
# Day 11

from sklearn.preprocessing import StandardScaler
import pandas as pd

print("=== Starting Data Preprocessing ===")

# 1. Remove non-vital columns
# The 'Attempt_Number' column is just a sequential attempt number and not a kinematic feature, so it must be removed
if 'Attempt_Number' in df_features.columns:
    features_only = df_features.drop(columns=['Attempt_Number'])
else:
    features_only = df_features.copy()

# 2. Handle missing values (if any)
# Replace any missing value with the column's mean to maintain data stability
features_only = features_only.fillna(features_only.mean())

# 3. Standardization
scaler = StandardScaler()
scaled_data = scaler.fit_transform(features_only)

# 4. Convert the data back to an organized DataFrame
df_scaled = pd.DataFrame(scaled_data, columns=features_only.columns)

print("=== Data Scaled Successfully! ===")
display(df_scaled)

=== Starting Data Preprocessing ===
=== Data Scaled Successfully! ===


,DT_t,DT_e,DT_s,DT_p,DT_a,FT_t -> e,FT_e -> s,FT_s -> t,FT_t -> p,FT_p -> a,FT_a -> s,FT_s -> s,FT_s -> Enter,Total_Duration,DT_o,DT_Backspace,FT_t -> o,FT_o -> Backspace,FT_Backspace -> p
0,-0.449881,-0.967205,2.007300,-0.482213,-1.045886,-0.336817,1.112160,-0.552714,1.543704,-0.034680,1.288381,0.576907,-0.165565,0.556543,0.0,0.0,0.0,1.136868e-13,0.0
1,0.039557,-1.160548,-0.141359,-1.485107,-1.026153,-0.403370,1.112160,-0.671798,1.905064,-0.222388,1.082151,1.469543,-0.066609,0.524868,0.0,0.0,0.0,1.136868e-13,0.0
2,1.655374,-1.158100,-1.427728,0.644325,-2.012838,0.209373,1.209864,-0.108533,0.000000,2.530283,1.290851,0.130589,2.898344,2.117779,0.0,0.0,0.0,1.136868e-13,0.0
3,0.562519,-0.382281,0.918835,0.781708,0.193390,0.386846,1.406464,0.093961,0.216958,1.061810,0.694389,0.152630,-0.080974,0.635830,0.0,0.0,0.0,1.136868e-13,0.0
4,0.616156,0.004405,-0.254447,-0.358569,0.884070,-0.463037,-0.688219,-0.873790,-0.355792,-0.785513,-0.791206,-2.547318,0.045647,-1.244347,0.0,0.0,0.0,1.136868e-13,0.0
5,0.099899,2.375916,-0.042408,1.715910,-0.138136,-1.442201,-0.503534,1.215468,-1.381090,-0.019801,0.495569,-0.332259,-0.213979,-0.335535,0.0,0.0,0.0,1.136868e-13,0.0
6,-0.543746,0.418012,0.890563,-0.454737,1.203756,2.650398,-0.980140,2.428423,-1.227728,-0.590937,-1.093759,0.598948,-0.680561,0.476459,0.0,0.0,0.0,1.136868e-13,0.0
7,-1.536032,0.205090,-1.441864,-1.581275,0.232858,-0.218246,-0.788306,-0.350220,-0.626341,-0.681358,-1.087585,-0.304708,-0.420935,-1.199325,0.0,0.0,0.0,1.136868e-13,0.0
8,-1.562850,0.604013,-0.254447,0.534419,0.848549,0.021190,-0.881244,-0.226613,0.008583,-0.943462,-0.889999,0.565887,-0.720463,-0.653878,0.0,0.0,0.0,1.136868e-13,0.0
9,1.119003,0.060695,-0.254447,0.685540,0.860389,-0.404135,-0.999204,-0.954185,-0.083359,-0.313953,-0.988792,-0.310218,-0.594906,-0.878392,0.0,0.0,0.0,1.136868e-13,0.0


In [6]:
# Day 12

from sklearn.ensemble import IsolationForest
import joblib

print("=== 🧠 Starting AI training ===")

# 1. Initialize the Isolation Forest algorithm
# n_estimators: Number of trees in the forest
# contamination: Percentage of anomalous data in training (set very low at 0.01 because all the data belongs to you)
ai_model = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)

# 2. Train the model on your standardized data (df_scaled)
ai_model.fit(df_scaled)
print("✅ Model successfully trained on your keystroke dynamics!")

# 3. Save the AI "brain" and the "scaling tool" as files
# These files will later be sent to the FastAPI server to make login decisions
joblib.dump(ai_model, 'keystroke_model.pkl')
joblib.dump(scaler, 'data_scaler.pkl')

print("=== 💾 Model files (.pkl) saved successfully in the working directory! ===")

=== 🧠 Starting AI training ===
✅ Model successfully trained on your keystroke dynamics!
=== 💾 Model files (.pkl) saved successfully in the working directory! ===


In [7]:
# Day 13

import joblib
import pandas as pd
import numpy as np

print("=== 🧪 Starting model testing and evaluation ===")

# 1. Load the model and scaler from the saved files
loaded_model = joblib.load('keystroke_model.pkl')
loaded_scaler = joblib.load('data_scaler.pkl')

# 2. Create a matching attempt (simulation of the genuine user)
# Take the mean of the real recorded values as a sample of the legitimate user
genuine_sample = df_features.drop(columns=['Attempt_Number']).mean().to_frame().T

# 3. Create a completely different attempt (simulation of an attacker typing very slowly or unusually)
# Multiply the timing values by 4 to simulate an intruder who does not have the same muscle memory
imposter_sample = genuine_sample * 4.0

# 4. Combine and prepare the samples for testing
test_df = pd.concat([genuine_sample, imposter_sample], ignore_index=True)

# 5. Apply the same standardization used during training
test_scaled = loaded_scaler.transform(test_df)

# 6. Pass the data to the model for prediction
predictions = loaded_model.predict(test_scaled)
scores = loaded_model.decision_function(test_scaled)

# 7. Display the results
results = pd.DataFrame({
    'Attempt_Type': ['Genuine (Owner)', 'Imposter (Intruder)'],
    'Raw_Score': scores,
    'Prediction': ['Accepted (1)' if p == 1 else 'Rejected (-1)' for p in predictions]
})

print("=== 📊 Test Results ===")
display(results)

=== 🧪 Starting model testing and evaluation ===
=== 📊 Test Results ===


C:\Users\ANAS A A KHAMAYSA\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
C:\Users\ANAS A A KHAMAYSA\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


,Attempt_Type,Raw_Score,Prediction
0,Genuine (Owner),0.137813,Accepted (1)
1,Imposter (Intruder),-0.069543,Rejected (-1)
